# P2a — BGE reranking of existing chunks (pilot, 2×T4)

Scores deterministic windows cut from the *exact* P1-submitted chunk text.
Outputs scores only, never replacement text or a BTC submission. Required
private dataset: `ptquanh/r2ai-p2a-chunk-pilot-v1`.
Optional Kaggle Secret: `HF_TOKEN`. Model: `BAAI/bge-reranker-v2-m3` (<15B parameters).

In [ ]:
%pip install -q --no-cache-dir 'transformers>=4.51,<5' 'accelerate>=1.6,<2' 'huggingface-hub>=0.30,<1' 'pyarrow>=18,<22' 'pandas>=2.2,<3'

In [ ]:
import hashlib, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SCOPE='pilot'
MODEL_KEY='bge'
MODEL_ID='BAAI/bge-reranker-v2-m3'
BATCH_SIZE=16
MAX_LENGTH=1024
INSTRUCTION='Judge whether this exact retrieved chunk contains evidence that answers the multilingual medical question.'
WORK=Path('/kaggle/working/p2a_chunk_pilot')
WORK.mkdir(parents=True,exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN')
    if token: os.environ['HF_TOKEN']=token
except Exception: pass

def sha256_file(path,block=4*1024*1024):
    digest=hashlib.sha256()
    with open(path,'rb') as source:
        while piece:=source.read(block): digest.update(piece)
    return digest.hexdigest()

assert torch.cuda.device_count()==2,torch.cuda.device_count()
input_root=Path('/kaggle/input')
pair_paths=list(input_root.rglob('p2a_pairs.parquet'))
chunk_paths=list(input_root.rglob('p2a_chunks.parquet'))
manifest_paths=list(input_root.rglob('p2a_manifest.json'))
if not(len(pair_paths)==len(chunk_paths)==len(manifest_paths)==1):
    raise RuntimeError({'pairs':pair_paths,'chunks':chunk_paths,'manifests':manifest_paths})
manifest=json.loads(manifest_paths[0].read_text(encoding='utf-8'))
assert manifest['schema_version']=='p2a_chunk_windows_v1' and manifest['scope']==SCOPE
for path in (pair_paths[0],chunk_paths[0]):
    assert sha256_file(path)==manifest['artifacts'][path.name]['sha256'],path
pairs=pd.read_parquet(pair_paths[0]); chunks=pd.read_parquet(chunk_paths[0])
assert len(pairs)==manifest['validation']['pairs'] and len(chunks)==manifest['validation']['chunks']
assert pairs.pair_id.is_unique and chunks[['query_id','chunk_ordinal']].duplicated().sum()==0
assert pairs.query_id.nunique()==manifest['validation']['queries']
input_path=WORK/'p2a_pairs.parquet'
pairs.to_parquet(input_path,index=False,compression='zstd')
print({'gpus':[torch.cuda.get_device_name(i) for i in range(2)],'scope':SCOPE,
       'queries':pairs.query_id.nunique(),'chunks':len(chunks),'windows':len(pairs),
       'control_hash':manifest['control_zip_sha256']},flush=True)

In [ ]:
WORKER = 'from __future__ import annotations\nimport argparse, gc, json, math, os, time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer\n\ndef parse_args():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--input\',type=Path,required=True); p.add_argument(\'--out\',type=Path,required=True)\n    p.add_argument(\'--model-key\',choices=[\'bge\',\'qwen4b\'],required=True); p.add_argument(\'--model-id\',required=True)\n    p.add_argument(\'--gpu\',type=int,required=True); p.add_argument(\'--world-size\',type=int,default=2)\n    p.add_argument(\'--batch-size\',type=int,required=True); p.add_argument(\'--max-length\',type=int,default=1024)\n    p.add_argument(\'--instruction\',required=True); return p.parse_args()\n\ndef write_part(out_dir, number, rows):\n    if not rows: return\n    pd.DataFrame.from_records(rows).to_parquet(out_dir/f\'part_{number:06d}.parquet\',index=False,compression=\'zstd\')\n\ndef load_done(out_dir):\n    done=set()\n    for p in sorted(out_dir.glob(\'part_*.parquet\')):\n        done.update(pd.read_parquet(p,columns=[\'pair_id\']).pair_id.astype(str))\n    return done\n\ndef main():\n    a=parse_args(); torch.cuda.set_device(0); device=torch.device(\'cuda:0\')\n    frame=pd.read_parquet(a.input).reset_index(drop=True)\n    qids=sorted(frame.query_id.astype(int).unique())\n    owned={qid for i,qid in enumerate(qids) if i%a.world_size==a.gpu}\n    frame=frame[frame.query_id.isin(owned)].copy().reset_index(drop=True)\n    a.out.mkdir(parents=True,exist_ok=True); done=load_done(a.out)\n    pending=frame[~frame.pair_id.astype(str).isin(done)].copy().reset_index(drop=True)\n    print(json.dumps({\'event\':\'start\',\'model\':a.model_key,\'gpu\':a.gpu,\'rows\':len(frame),\'pending\':len(pending)}),flush=True)\n    tokenizer=AutoTokenizer.from_pretrained(a.model_id,padding_side=\'left\' if a.model_key==\'qwen4b\' else \'right\')\n    revision=\'unknown\'; prefix_tokens=suffix_tokens=None; yes_id=no_id=None\n    if a.model_key==\'bge\':\n        model=AutoModelForSequenceClassification.from_pretrained(a.model_id,torch_dtype=torch.float16).to(device).eval()\n    else:\n        model=AutoModelForCausalLM.from_pretrained(a.model_id,torch_dtype=torch.float16,attn_implementation=\'sdpa\').to(device).eval()\n        no_id=tokenizer.convert_tokens_to_ids(\'no\'); yes_id=tokenizer.convert_tokens_to_ids(\'yes\')\n        prefix=\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n<|im_start|>user\\n\'\n        suffix=\'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\n        prefix_tokens=tokenizer.encode(prefix,add_special_tokens=False); suffix_tokens=tokenizer.encode(suffix,add_special_tokens=False)\n        if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token\n    revision=getattr(model.config,\'_commit_hash\',None) or \'unknown\'\n    started=time.time(); buffer=[]; part_no=len(list(a.out.glob(\'part_*.parquet\')))\n    with torch.inference_mode():\n        for start in range(0,len(pending),a.batch_size):\n            block=pending.iloc[start:start+a.batch_size]\n            queries=block.query_text.astype(str).tolist(); docs=block.passage_text.astype(str).tolist()\n            if a.model_key==\'bge\':\n                inputs=tokenizer(queries,docs,padding=True,truncation=\'longest_first\',max_length=a.max_length,return_tensors=\'pt\')\n                token_counts=inputs[\'attention_mask\'].sum(1).tolist(); inputs={k:v.to(device) for k,v in inputs.items()}\n                raw=model(**inputs).logits.float().reshape(-1); norm=torch.sigmoid(raw)\n            else:\n                formatted=[f\'<Instruct>: {a.instruction}\\n<Query>: {q}\\n<Document>: {d}\' for q,d in zip(queries,docs)]\n                encoded=tokenizer(formatted,padding=False,truncation=\'longest_first\',max_length=a.max_length-len(prefix_tokens)-len(suffix_tokens))\n                ids=[prefix_tokens+x+suffix_tokens for x in encoded[\'input_ids\']]\n                inputs=tokenizer.pad({\'input_ids\':ids},padding=True,return_attention_mask=True,return_tensors=\'pt\')\n                token_counts=inputs[\'attention_mask\'].sum(1).tolist(); inputs={k:v.to(device) for k,v in inputs.items()}\n                final=model(**inputs).logits[:,-1,:].float(); raw=final[:,yes_id]-final[:,no_id]; norm=torch.sigmoid(raw)\n            for row,raw_score,norm_score,tokens in zip(block.itertuples(index=False),raw.cpu().tolist(),norm.cpu().tolist(),token_counts):\n                buffer.append({\'pair_id\':str(row.pair_id),\'query_id\':int(row.query_id),\'doc_id\':int(row.doc_id),\'passage_rank\':int(row.passage_rank),\'raw_score\':float(raw_score),\'normalized_score\':float(norm_score),\'input_tokens\':int(tokens),\'model_key\':a.model_key,\'model_id\':a.model_id,\'model_revision\':revision,\'gpu_shard\':a.gpu})\n            if len(buffer)>=512 or start+a.batch_size>=len(pending):\n                write_part(a.out,part_no,buffer); part_no+=1; buffer=[]\n            if (start//a.batch_size+1)%50==0 or start+a.batch_size>=len(pending):\n                elapsed=time.time()-started; complete=min(start+a.batch_size,len(pending)); print(json.dumps({\'event\':\'progress\',\'model\':a.model_key,\'gpu\':a.gpu,\'complete\':complete,\'total\':len(pending),\'pairs_per_second\':round(complete/max(elapsed,1e-6),3)}),flush=True)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    all_parts=[pd.read_parquet(p) for p in sorted(a.out.glob(\'part_*.parquet\'))]\n    scores=pd.concat(all_parts,ignore_index=True).drop_duplicates(\'pair_id\',keep=\'last\')\n    expected=set(frame.pair_id.astype(str)); actual=set(scores.pair_id.astype(str)); assert actual==expected,(len(expected-actual),len(actual-expected))\n    scores.sort_values([\'query_id\',\'doc_id\',\'passage_rank\']).to_parquet(a.out/f\'{a.model_key}_gpu{a.gpu}.parquet\',index=False,compression=\'zstd\')\n    (a.out/f\'{a.model_key}_gpu{a.gpu}.json\').write_text(json.dumps({\'model_key\':a.model_key,\'model_id\':a.model_id,\'model_revision\':revision,\'gpu\':a.gpu,\'rows\':len(scores),\'seconds\':time.time()-started},indent=2),encoding=\'utf-8\')\n    print(json.dumps({\'event\':\'complete\',\'model\':a.model_key,\'gpu\':a.gpu,\'rows\':len(scores)}),flush=True)\nif __name__==\'__main__\': main()\n'
worker_path=WORK/'p2a_worker.py'
worker_path.write_text(WORKER,encoding='utf-8')
print({'worker_chars':len(WORKER)})

In [ ]:
processes=[]; logs=[]
for gpu in range(2):
    out=WORK/'checkpoints'/f'gpu{gpu}'; out.mkdir(parents=True,exist_ok=True)
    log_path=WORK/f'bge_gpu{gpu}.log'; log=log_path.open('w',encoding='utf-8'); logs.append((log,log_path))
    cmd=[sys.executable,str(worker_path),'--input',str(input_path),'--out',str(out),
         '--model-key',MODEL_KEY,'--model-id',MODEL_ID,'--gpu',str(gpu),
         '--batch-size',str(BATCH_SIZE),'--max-length',str(MAX_LENGTH),'--instruction',INSTRUCTION]
    env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu); env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
    processes.append(subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env))
while any(process.poll() is None for process in processes):
    time.sleep(60)
    status={'returncodes':[process.poll() for process in processes]}
    for _,path in logs:
        if path.exists(): status[path.name]=path.read_text(encoding='utf-8',errors='replace').splitlines()[-1:]
    print(status,flush=True)
for log,_ in logs: log.close()
for process,(_,path) in zip(processes,logs):
    if process.returncode!=0:
        raise RuntimeError(f'P2a BGE worker failed ({process.returncode}):\n'+path.read_text(encoding='utf-8',errors='replace')[-8000:])
scores=pd.concat([pd.read_parquet(WORK/'checkpoints'/f'gpu{gpu}'/f'bge_gpu{gpu}.parquet') for gpu in range(2)],ignore_index=True)
assert len(scores)==len(pairs) and scores.pair_id.is_unique
assert set(scores.pair_id)==set(pairs.pair_id)
assert scores.model_revision.nunique()==1 and np.isfinite(scores.raw_score).all()
print({'scored_windows':len(scores),'revision':scores.model_revision.iloc[0]})

In [ ]:
scored=pairs.drop(columns=['query_text','passage_text']).merge(
    scores[['pair_id','raw_score','normalized_score','input_tokens','model_revision','gpu_shard']],
    on='pair_id',validate='one_to_one')
pair_output=WORK/'p2a_pair_scores.parquet'
scored.to_parquet(pair_output,index=False,compression='zstd')
aggregated=scored.groupby(['query_id','chunk_ordinal','doc_id','chunk_sha256'],as_index=False).agg(
    bge_chunk_score=('normalized_score','max'),bge_chunk_raw=('raw_score','max'),
    windows_scored=('pair_id','size'),max_input_tokens=('input_tokens','max'))
chunk_scores=chunks.merge(aggregated,on=['query_id','chunk_ordinal','doc_id','chunk_sha256'],validate='one_to_one')
assert len(chunk_scores)==len(chunks) and (chunk_scores.window_count==chunk_scores.windows_scored).all()
chunk_scores['bge_chunk_rank']=chunk_scores.groupby('query_id')['bge_chunk_score'].rank(method='first',ascending=False).astype(int)
chunk_output=WORK/'p2a_chunk_scores.parquet'
chunk_scores.to_parquet(chunk_output,index=False,compression='zstd')
validation={'schema_version':'p2a_chunk_scores_v1','scope':SCOPE,'model_key':MODEL_KEY,'model_id':MODEL_ID,
 'model_revision':str(scores.model_revision.iloc[0]),'queries':int(chunk_scores.query_id.nunique()),
 'chunks':len(chunk_scores),'pairs':len(scored),'nonfinite':int((~np.isfinite(chunk_scores.bge_chunk_score)).sum()),
 'control_zip_sha256':manifest['control_zip_sha256'],
 'input_pairs_sha256':manifest['artifacts']['p2a_pairs.parquet']['sha256'],
 'input_chunks_sha256':manifest['artifacts']['p2a_chunks.parquet']['sha256']}
(WORK/'validation.json').write_text(json.dumps(validation,indent=2),encoding='utf-8')
run_manifest={'schema_version':'p2a_chunk_reranker_run_v1','input_manifest':manifest,'validation':validation,
 'aggregation':'max normalized window score per exact submitted chunk','max_length':MAX_LENGTH,
 'outputs':{path.name:sha256_file(path) for path in (pair_output,chunk_output)}}
(WORK/'run_manifest.json').write_text(json.dumps(run_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
output_zip=Path('/kaggle/working')/f'p2a_bge_chunk_{SCOPE}_scores.zip'
members=['p2a_pair_scores.parquet','p2a_chunk_scores.parquet','validation.json','run_manifest.json']
with zipfile.ZipFile(output_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=9) as archive:
    for name in members: archive.write(WORK/name,arcname=name)
assert zipfile.ZipFile(output_zip).testzip() is None
print(validation); print({'output':str(output_zip),'bytes':output_zip.stat().st_size,'sha256':sha256_file(output_zip)})